# BatikCraft — Kaggle Asset Pack Builder (Standalone)

Notebook ini memproses dataset batik menjadi kandidat modular, membuat contact sheet dan
`review.csv`, lalu mengekspor `.batikpack` yang dapat langsung dipasang di BatikCraft Studio.

Pipeline bersifat **semi-otomatis**: computer vision membantu crop/segmentasi, tetapi hasil
wajib dikurasi manusia sebelum menjadi pustaka bawaan.

**Notebook ini tidak memerlukan repository BatikCraftStudio.** Seluruh helper ekstraksi,
pembuat `.batikasset`, pembuat `.batikpack`, dan validator sudah tertanam di notebook.


## 1. Siapkan dataset saja

Tambahkan dataset gambar batik melalui tombol **Add Input** di Kaggle. Tidak perlu:

- clone repository;
- menambahkan repository sebagai Kaggle Dataset;
- GitHub token;
- file `src/batikcraft_studio`;
- file Python tambahan.

Cell berikut otomatis menulis helper tertanam ke:

```text
/kaggle/working/kaggle_asset_pipeline.py
```


In [ ]:
from pathlib import Path
import base64
import hashlib
import importlib
import subprocess
import sys
import zlib

# Kaggle umumnya sudah memiliki paket ini. Instal hanya bila ada yang belum tersedia.
required = {
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
}
missing = []
for module_name, package_name in required.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(package_name)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# Pipeline lengkap disimpan di dalam notebook ini. Tidak perlu clone repository,
# GitHub token, atau Kaggle Dataset yang berisi source code BatikCraft Studio.
EMBEDDED_PIPELINE_SHA256 = "9467d2cc8beda50e077daa503729ef196afd819146eb4b76a78d1007b3005acc"
EMBEDDED_PIPELINE_B64 = (
    "eNqtW3tz20aS/1+fAkHV3QIWCJGUpZXpYGtlW3ZccWyX7L27XYXLGhJDEiEIIAAoklb0ufb/+2T3654ZPPiIna1zKgIwj56efnfP"
    "0LbtT6VIQhGnibReiDJavMzFtLR+FLNZLK0symQcJfK5laRWLrO0iMo031rRMkvzEi2/rqJchr5t2yfTPF1ao9F0Va5yORqZMSJJ"
    "0hKA06Q40U1jUcjLp541Ke49ay6KeRyNPeuXIk08aynKuQfAnvUlyqZRLBXcUJRiEouikIUBXDWpEVFqOl5sS1m8/aCaM8ADeNP3"
    "kcF/BIYfsZcNfRqsJvd9z0pWy2xricJKMg9zQZmCvrJQQfv49p2B9HYpZsCSH69ysdavH7Li5OTj9csfR68/3P50/RmL0cenlz/c"
    "/HRtBZY9JiJPiMgd2k7ZycRkYXuW3fO79sn1p083n6up6uv43Gray+vPN28+3L69+YRhjr1My2jaydJFypCjQiYd+kMfaZ6IpXot"
    "5aIAs+g1FlGSbIXtnrz96frNzejmfz4TqAfbz5IZDfB/ycxTqpe1HGf8Ml6qJ9Y0z6n9ePLjzc1HBtHjtfKVpOdWFvygPwspMww8"
    "OWE2Wh+1sN3keZo7t6ukjJbqwx2cWPiXEbNPTv5acd4BU77IJPgM4K6GcrMpczEheXuZJtNopqbSFBBslKdpOWAp4OZ1mi9224gf"
    "oygcWEWZ7xA9lFOxissOBCoX+bZz37PrKUTWalJDlV6pSdY7Nakx417mBfCsJoGVxMyqX6zKeZo3YMZbYd2mk/lShFFjXCiLSR5l"
    "ZRPWNcmHtUzDVSxyUrIotojduVgUEciRWItVLviVKWPxNn0FVVY0HAGCLAZWucpieYfpnuX7/pClbLqKY2LjJIVGJDIpmbOzPAoh"
    "RgRmKTajIl3lEzkqohDEiZISM3tX3S7301D0fKl6Lnr9uiMFeWKRDaxpnArq7fr9iwruBLoZAXdZjDKZjyLSPQPmqQIP8qUjMZnI"
    "rBxNSBZCmUxkE97VpYZXlIDBKtXCp9ftP+UR5Xy1HCciitvdz/q7+2Q8CjXgN+s9WdWAHydKDOUUJhJmtBxFSVSORk4h46kWb/qX"
    "jn+Rk9IfjYCKKMtcjwBdmxIMOpO4cpff7HDdb4BUCX0LTNX6TTC0kgBCIabYdqiA6GaAYBh/zfIU3Cm31eYrtvFSevcw9zBDidXG"
    "wzqDaFVcto8BTBMYA7BtLjUNvg5UTenwlKOAc3kfyfUIPuqrANVQH0OPAUtXZbYqR0Ser0Gb2g9NUj76rJjsJI4Avxcx0Yi0lVx0"
    "/nUK1DM6aoZPztf+qm19adihBFYpjDaV3FLZQP6aYOQM4ULdUorZvjFRhiyZjchXNwyxVinVaiAYezIX/YvLupnMVP01Hqcbsww0"
    "0bNaf9SCexYBuydqGnkGjbBxtdESBhF6nEu/WI2d3L77p+h86Xae+aPO8JSMXocUocz1JB+vUea4/gSKOU3jEK+gcywm0rEtNV5r"
    "mYZMDz3L7vi2ezfo9bsKz2iK2KvkEeCogBvfcZT2W+wCPjeSuVWCOwtrnMZybi1S8HTma1usZYHA6I1OoyQcNY2HU2tbQCZLb75u"
    "BaKNjzS3nMpS2GcLjhjPogSCfqbBFmdzsYSviSBvi3khkjMW5g7HJ/AWx6Z/2yjgDyyLSCSdAxPUtqfAEsEZ2N5AvTa3JFvYFVtB"
    "DKstH6hOfX5UjMIod1yEsSH+3zoZRGA6jTZ+nK4ldQByI2Si9TJq49n5LE7Hjv3EdhkgQaOQ1nEbFr/BnMxIPuupRqu9adtsawV8"
    "AuuukhGaUmFbQyd8QsJH9cPN7QzQ08MDc82/BAsVq+U3bz38nX3vASfxHqj9+CLLZBI6DtKAUA9FN/fVxgzWSoYON3rWQm6DWCzH"
    "obCg8E5nc9cdKk3c3PWGruviGy+KqKw7r4HH+7R8TfO1/rxqRkDWWMYr5BtRCbBivhCJb71ZJQIvJim6DkPrLTGDdEvpkvb7DvvP"
    "wR4PubmpzW1uHVTraYWXUmpgJJcroDGwHmj6445ik7FzNHWcrOYGL3WUISzXv89Yt0XmjK2sk7lmsFvRIJcidDJOsDgignXVtGDq"
    "kPW47/vRMpQTWGsnyXzKqRgPNQstK9jnK6xII9/+dHtz/Wr08sO7D7cV9RSoqOCgylDuv8jsGrJ9VuQSgGktJZg4EdYDLWAohpXW"
    "QIYh+XAkmbwb9IfPrWIiYsJyGSUOQvF6F3CZeHVoXi2WavT3Vm/Q2l4uKUJ0IpUdgvOQM2dtPVHjsTPVMq9aXPZJMs/SmD1ywFt/"
    "//nmdnSN/beYzFANuRG1LtQ6msqxGGskJvflyzQGNTQa1MZkHL14c9t/d/1CQR2neSgpWwDh4Q7JXSf433EAiRWJnp2efhl4Vrd+"
    "7ZB6KSgzBWEpQ5hiRwH1EVJvM8Vkcq7nfexTbKIi6KpZYVSUah5EXsQzP0nzJS18YKbVwSJ6el9Np9Ea8RhOk6EpLvEkbvDpk+jb"
    "c0Ht/sUFNuDR022soQSOYY6I48VC07Ccg5Vzct60lpmqaPn5h9ubTz+MXrx9f337d+u02fbh86e/KXAynCmXScQXyK6dr7Pmze31"
    "34HwJRbrXXZNFlXhNI7KNeR9hOnUqmaHUUxM4+VYh8gpOs65Z527DZ3S3CJQRJdzI4kk7TuqQOTyu91LFylMbw+JZZpnoEs6295s"
    "Gmj89OH24w+jl+8+fLppYIHORQuLPwTtw8eb97+3JW3XoTZGiFdJOUrS5IvMU4bmslQUC5+0sqlKaimihN8FV3uQMTGGAWdgHcs/"
    "v6gNG6JJLK8ZpnGsoke8T2daA49ZFoT/LWetUmcOSqYzfz/bxvjKGZo02+myDK49aw4a+Jf92hQ1M/CjQCvPC+86Ih8pSojLyFiM"
    "NEmQ6snwZQXqv6Ny/olGabZcuc8tARuPGWS91hVAWr2o92fCjoi9j0hm0umBaTuBxcaztp41xnbGoBoBZYzuIpDLcFSAebTibrAy"
    "XsPuPr2iCHQ8r17Jr0Jqr6zvAw0BL/6f+wPOEaNkJVtwMkEBlNYBiIFCRYn+VTtOiT2r1GoDDmw6mOp65nPLn+1wDpC0HwHQzel4"
    "faqnoAW73Z6O56d7s4pJmhv34z97Bh4/vYBtob9PLHGmcHwCasGcobn3hAbyRs/8PhoJ9VosW6CZQ5VEObQfqH8Hz3GnBF68dCM4"
    "U+MpmHAarv/LwPpC8VR/+ISf50Mq1FIRS+ossaobQH4hgEp+W+UhqJJZjoWk/mbBpXVrseZS0jGB5sjlsM6fVRpv/QWaclFLXhlV"
    "Hp5gVgUorVeIAUqZaVb3jIWkSU+cXqeaoStTboNiG1KAGE7HURKvogeCse7Q/NMekRnAXSyyPTp4vju4mZBsCnLG1neBhjlAi2Gq"
    "amkN31bD53r4tho+3xneYhiTHQZn4209GsZ/yOZcXCmmbYkn2AS9b+h9U7TCFEC7GxC1jhXqhsaymgoBB4Am2U4XMiEKIfZ1KOXO"
    "4qhsJN2ccXMMSnMaOXblXNgaPdRpY13/Ruxsq0K4PZETucCzEGukVZ69WkiYWXsmYgGuUEdURDRgggByjmcZIT2wHxvpqKmfE1RK"
    "rFe5pHGmqm5PxTiPJnhZiIghSqgrnkkKHPBcYq1lC6IpzhNEFUhhWBYls1lEb+VqmQnCcppjGJ7zSECr6pq+XaxiLK1BPjaZkgBB"
    "Byl+yWzDk8P6gthHFKPccFk4nB9oDvwnDwDjW+cJxitqfTQVBKf2gsT5dp3PXy4o08lgzGEG2Fh4AAANGKWLhu3gmXsFvW+drUpD"
    "xHuTjhG8VnnUmBYjnDtV2xqCfqvFuDVSObpVEv1KqBRSJuz+PBbZuvDA+ZeGVNugMAIE8if65MtXtSyk15RAjcZ0egVh9udyo4Y6"
    "LbXW08nbEXSsPeC/vghDR3XCxCjcKpOfebrH6CnkpFRnb7LQuGv3za47CSXssk7L9FRaTiarJQmxdMzmew2/TukZzbSCAAGVgWP9"
    "BwJnaurWTfiKZaKBIAHOcgR0yN1+ksssT+mQ74EHPp49NMY9DlQi55O0m2yOTQbVGE0W1sxDd7jHKenzRghHwxt5VAVQbujUwLrh"
    "B1wOnQGizSBqv4JFXMP1DmzPqjAioZwA/F64MeXoBPQkx6UN69qbV463Tdlj4aYKNNsU5ypsnM0F+6xicbcdbE+x781gc7oeEkNo"
    "SfIB2pnKGOkypVYIKh3EImvXo6ymnQhVccxsLA5mlO1ldpOY2zcvrkEGmo1MceCdDzWQN2JVULnuBWyUw2h7iOkR0nfd3RBvz7E3"
    "oLlnjQ/l5r9H0Na/OBLpzXKxbWfnrR1VwAbnra1gGzofc3p/9nqXrvf1PN38Q6JGakUrA3RvMEQgoj8GcMrPrWmmi2okTveu9f33"
    "iJdZ9bz7tjgQKGTConT3iET1SAeQ/mnB/7mUHY6YaLDiCH0vGB56CJ7S9LvORbfbHQzdI5RSo4zVmGYU8UcUJpui+NR+2PHZjx2l"
    "kghXlnhnrR10L0K8k+jhkaSDbj9sqquOOIgne65CnYAIOvigk2fkThB/BLRV9QjvVD0yB9N7rPRaHLzuQyKv90lH5jNdHCEDlvDL"
    "lAtTQLM9l04wqGDP5bYwmpRcxCJH6dwp18p2+Q/FLYQQmbnS/UtwPrzrXEJeTq07ot9wB3VluQ2HqvMYBwTTlogYUZ82jBATWLbr"
    "I3ihct/p1LbAkEdgssNHj3fmEV9qLAGHhP1ejsp035m6xjV41p5hU5GzqsUoI+c9dd0DFUu1IRNUrPOI5EAdvREDYAcpLmkm2Xm6"
    "Zpf1oO4PDEiDNn6d+fwlIEwPH/66nm0OrezBxjfvCMkonkIL23HbkIZazDuiLxDIHti/2f4vKRKIjU8NgNg4p6IJjU/PrpdmWHXV"
    "AEEgnWoObDBoJ7WhkUxQm86xMMTTKyJOc+jUbONTh1svrSKIxuKq4bGO0JmK5uArXi05wL5TFGzQRBOiQQG97fYuW/syW9nfh97A"
    "Lpo6zqAQFkhkoU+F7tf06RBvPYNhoJ+wQjwYOsnHscTe+nTWY5MTvBbwbLqO1wohmxJ0ILgkR6XkkdeoEpODIGgCo1cElx4LYnDh"
    "cVIZ9J51G4X/QyFrO2J9biEboh0WTwhOHTNyedsUTug2lD+RUeyQfWAs3DNMbB4gMYoBXz7yE7l2bJg923MYNKdvBP8JZ7GnT/uQ"
    "Gqf/9MLrnz/z+v1nlI6GuVgH1T0mn/44DNRt13M8Wr/tnBijO8L5CbAaOPSG7JU+hrtnXN4kCKN7CIYTeYQcVo7ltPTKNAsmGtMa"
    "zdbcdVTO1fUqP800IXxzYuxScFbkHJyxVAT4IE4jSy+ZGtc7vkeP86tbHQ4v2+ldevrpemq1W1mIJex4MvPfXb9/+Y8Pn4A1FYXB"
    "uzbFr0HyRrLsUFTV+H/HjCsQPsdAIy6QFNiToxHzND4Gz3UUYptnZ32v3T6X0WxeUgexkVnmZ3SXxdHwm0SwgZSh9w42JAB+yfkh"
    "jTi9oDGnzIm+68EdM72Nkbgb9C+Gjz8nqtUYikfrf/9l6ab6ZN3vT+FrYMjjwHna986feiRybdH1C3Ev+YjsbNq+m0GhBcnToHuO"
    "cIAvoP26EvCq2+BZ10sRmS8RzekMsKooIGKMJiLWGYCq7jCnudKjpXJXotqCxEFuYO7z+dDc6QimLSnAJ+lgiHtQvmDsAlX7ncly"
    "MhdJImPHRi99kyl0WoeAfEnh8NGfur2llqRyxKHERyGp1pvkaeaQT3iui88N0VaumPb+xL+6cr2d798RdXP8fy+KA9LOlGWSek7X"
    "4/9IDNX4PdFW6YzDszoKx4ZYN1qbQq2oupoG+kqnU8NnoUGXZ398/8beFYbqzlbNRJYsqvvxfGhSJRJeQzr+Pb0/DAp2FvJ+fu71"
    "YVjYBjw/pvg8jdS+mt7h15bq7/btUaocNwl10AYoupXjg2SrXCHoSiLLt1kc1yvHjS+taONVFIejyYrcQKguNsHBwjnqltpBN++S"
    "aAPNh+L7I6k+0HbulIBwDCfDQNcnWHO1AWTV5YQAnArsVTntXHWKCHYC7CF9CmybtXravhEBj8hl5eLef4UQ/lYKUMqZ7l+LoEAY"
    "g2n7jgmWbPfQBR9TkKFbr0fSCsTogUmkAPSuDruGFA6AJsF+PnTWzIYOZTHmmoo6zz9mUKpM4cBtgubZeAM4o1jFgUOzwfqq8bG1"
    "flRzIr0UXzbjSwvruz8ZeH8a7q5o+FzVrCilwRyPTXPLcJqhR+5DveAbHCIU9ZUfC+5oRndewcKgZ+5EEbWre3HfErKBZhnzqHGf"
    "zyeB5Mu/yEnt091e6ji1/XKZ2Vx5SqIpMqaWMOuL7v4/ooxupzi0imevba/qePtx9Orm9TtQ/pVHRgNJUxHLexkHlyzeX9ri3aQc"
    "sayiVzsKSmbKXAW7TlNVzXbuxHJr+x4saWcWTJUcF2dKTvkWjbqdDoJRdzXJDNkT5aUsRfDQTqQqtWsnHghjWklVNayVkdAonWBV"
    "A0yWYu/nTbuVWJX6HinGIqsh2ydzZGaNu+HqllanqH5OYVcDR8VkLpcC4+my/mNb7IlS2Dv4hmDfHrR+AGCrieaiuOlVPwgwSSvr"
    "Kb8Om5nrjvpCoMiV2IPDzLWVMznaTRyi5N8e0JtnU/CtfsZhD9TTH18+1bUZdLq+vuXjtrf7xecUn+yqyDy6deqHq2VWOLyYJ5OC"
    "fjwiikkUqZSO07ukDPoeHUeOqNCitBHC1wBWZkqW2zbFqJuxKQ825f2iTvi/lXakhpgItayE2R6UmakJ3JnLm26jAmTkTufP5Hu5"
    "ImT/pm5hVXP+f3nzWJMgTCcNyWr+KGVPsBo/UvH4crc9UMSiNfVVZEO0qkkVSyoYVbtu8Wz1A4ZGj2rw7MYvFhq9jdZHXZQAcQ0T"
    "a0Fq8N2uWMw3mJsShd3/IXnS93GXWVU927HlHB/RbelA35+Wmg0m+tkZTC17d7MV6iPOthrIqs4KOdfbiWrsOjjbWcgkPhxZkOVq"
    "3VzOgtZvnZz6knLlUfk2oBgXabwq6UYgBNj++Wc+d69HUyuVm7mY9EBFK5/+9+1GjSkDF/OyOBqDEAI6JhBLjjwY9s5FxgyJJv1C"
    "ItpUseYhcjcOjDly4t1x43HHWiV5DX9JMlwEX9hP85F867CNe9VJxK6oud8FvSMBSGtgMwry7RZ0qunQiaGu61XRpMfLui6WoCHq"
    "68haTNQZ+ZwmcNJ8lq44FWHhfGFvtrcD93nrYDGdKIOlDQZWb5gMEoBqxI71MCOV/TiC6Ceewz9YOkYSucn4DhKMVhvTR6S1YRHU"
    "p6tVTUpXpCrMtNHw7nbLT83gm2bd2Rx37x2pRHxcgtWOXbZ/pVz2Ps0ZQljwOSzAcHDkIQBqaKZamJ3J0PX2OmrfQgmBocZvwQMD"
    "etzDNTMxOUsIcahsNx3TRNIGvQ3FCojZbEG/teIAbTcy5xBlX5wEX2aBsw/qCEB7fZ5y14wShp7R4p27QoeqMSaLpUCCFVb9vOvQ"
    "5XF91w6O8LtAk1E506GnvrTzHB41S3+jH8IlmhyFiMX8MBl08qE3X2aUBZK9I+Sdsf3z5uoZ5dRH1/lsuIvcmW6TY/D+QpRzQo6V"
    "yn8XGBk4qvxQpqVcRnG0iKyp4mq096OPB/WDH3vASU31260BtOZOOfuhUgfdp9x8q9fESMoCs0G0B2SceJTWuiFF1uCECpvtgT6P"
    "EtBaPHS08m/F3I8n/wcAK00H"
)
pipeline_source = zlib.decompress(base64.b64decode(EMBEDDED_PIPELINE_B64))
if hashlib.sha256(pipeline_source).hexdigest() != EMBEDDED_PIPELINE_SHA256:
    raise ValueError("Embedded pipeline notebook rusak: checksum tidak cocok.")

PIPELINE_PATH = Path("/kaggle/working/kaggle_asset_pipeline.py")
PIPELINE_PATH.write_bytes(pipeline_source)
if str(PIPELINE_PATH.parent) not in sys.path:
    sys.path.insert(0, str(PIPELINE_PATH.parent))
importlib.invalidate_caches()
sys.modules.pop("kaggle_asset_pipeline", None)

import pandas as pd
from PIL import Image
from kaggle_asset_pipeline import (
    ExtractionConfig,
    build_curated_pack,
    extract_dataset,
    find_dataset_root,
    validate_asset_pack,
    write_review_files,
)

print("Standalone pipeline ready:", PIPELINE_PATH)
print("Pipeline SHA-256:", EMBEDDED_PIPELINE_SHA256)


## 2. Konfigurasi dataset dan output

`find_dataset_root()` mencoba path dataset BatikCraft yang umum, lalu mencari folder gambar
terbesar di `/kaggle/input`. Isi `DATASET_ROOT_OVERRIDE` hanya bila ingin menunjuk folder lain.


In [ ]:
DATASET_ROOT_OVERRIDE = None
# Contoh override:
# DATASET_ROOT_OVERRIDE = "/kaggle/input/batik-motifs"

DATASET_ROOT = (
    Path(DATASET_ROOT_OVERRIDE)
    if DATASET_ROOT_OVERRIDE
    else find_dataset_root()
)
WORK_ROOT = Path("/kaggle/working/batikcraft-asset-builder")

config = ExtractionConfig(
    dataset_root=DATASET_ROOT,
    work_root=WORK_ROOT,
    pack_id="batikcraft-default-library-v1",
    pack_name="BatikCraft Default Library",
    pack_version="1.0.0",
    pack_author="Balya Rochmadi",
    pack_description="Asset modular hasil ekstraksi dan kurasi dataset batik.",
    extraction_modes=("full", "components", "grid"),
    max_source_side=1800,
    grid_size=512,
    grid_overlap=0.25,
    max_candidates_per_image=40,
    auto_accept_confidence=0.86,
    master_asset_size=1024,
    thumbnail_size=192,
    max_source_images=None,  # ganti angka, misalnya 100, untuk uji cepat
)

print("Dataset:", config.dataset_root)
print("Work root:", config.work_root)
print("Output pack:", config.output_pack)


## 3. Ekstrak kandidat modular


In [ ]:
candidates = extract_dataset(config)
print(f"Kandidat tersimpan: {len(candidates):,}")
print("Folder candidate:", config.candidate_root)


## 4. Buat antrean kurasi dan contact sheet


In [ ]:
review_df = write_review_files(candidates, config)
display(review_df.head(20))

contact_sheets = sorted(config.contact_sheet_root.glob("*.jpg"))
print("Review CSV:", config.review_csv)
print("Contact sheets:", len(contact_sheets))
if contact_sheets:
    display(Image.open(contact_sheets[0]))


## 5. Kurasi manusia

Edit `review.csv`:

- `keep=1` untuk diterima, `keep=0` untuk ditolak;
- perbaiki `name`;
- pilih `category`: `motif-pokok`, `isen-isen`, `ornamen`, `tekstur`, atau `lainnya`;
- pisahkan tags dengan `|`;
- tambahkan catatan pada `notes`.

Kandidat confidence tinggi hanya diberi saran awal `keep=1`; tetap periksa contact sheet.
Setelah kurasi di spreadsheet, upload kembali CSV final dan ubah path di cell berikut.


In [ ]:
CURATED_REVIEW_CSV = config.review_csv
# Contoh bila CSV hasil kurasi di-upload sebagai Kaggle input:
# CURATED_REVIEW_CSV = Path("/kaggle/input/batik-review-final/review.csv")

curated = pd.read_csv(CURATED_REVIEW_CSV)
accepted = curated[
    curated["keep"].astype(str).str.casefold().isin(
        {"1", "true", "yes", "y", "keep"}
    )
]
print(f"Disetujui: {len(accepted):,} dari {len(curated):,}")
display(accepted.head(20))


## 6. Bangun dan validasi `.batikpack`


In [ ]:
pack_path = build_curated_pack(
    config,
    curated_review_csv=CURATED_REVIEW_CSV,
)
validation = validate_asset_pack(pack_path)
print("Pack selesai:", pack_path)
print("Ukuran:", f"{pack_path.stat().st_size / 1024 / 1024:.2f} MB")
print("Validasi:", validation)
print("Laporan validasi:", config.validation_report)


## 7. Install di BatikCraft Studio

Unduh file output:

```text
/kaggle/working/batikcraft-asset-builder/batikcraft-default-library-v1.batikpack
```

Di aplikasi pilih:

```text
Asset → Install Asset Pack…
```

Paket kemudian muncul di **Pustaka Asset** dan dapat dicari, dipreview, serta dimasukkan
ke canvas tanpa menjalankan Kaggle lagi.


## Catatan troubleshooting

- Bila dataset tidak ditemukan, gunakan **Add Input** dan atur `DATASET_ROOT_OVERRIDE`.
- Bila ingin uji cepat, set `max_source_images=100`.
- Notebook tidak akan mencari `/kaggle/working/BatikCraftStudio` dan tidak akan meminta clone.
- Internet hanya mungkin diperlukan bila environment Kaggle tidak memiliki salah satu dependency
  Python dasar; repository BatikCraftStudio tetap tidak diperlukan.
